# Day 7: Deploy your first LLM-powered app on Streamlit Cloud and integrate basic logging

## 🧠 Core Theory (Just-in-Time)

As you transition into AI Engineering, getting your models out of local scripts and into a usable interface is critical. 

### Why Streamlit Cloud?
Streamlit Cloud allows rapid deployment of Python-based UIs directly from a GitHub repository. It abstracts away server management, Dockerization, and reverse proxies, allowing you to focus purely on the application logic. For AI prototypes and internal tooling, it is the industry standard for fast iteration.

### Why Structured Logging?
In software engineering, `print()` statements are insufficient for production. In AI Engineering, this is doubly true. LLMs are non-deterministic, meaning the same input can yield different outputs. 
- **Traceability:** Logging captures the exact prompts sent and completions received.
- **Observability:** If an API call to OpenAI fails or times out, structured logging (e.g., using Python's built-in `logging` module) ensures you have timestamps, severity levels (INFO, WARNING, ERROR), and module names to diagnose the failure quickly without exposing raw stack traces to the end-user.



## 💻 Code Implementation

Below is a production-grade Streamlit application. It emphasizes strict type hinting, proper exception handling, and robust logging. We use LangChain's stable `ChatOpenAI` wrapper for interaction.


In [ ]:
import logging
import os
import streamlit as st
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

# 1. Configure Logging (Production-Grade Setup)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    handlers=[
        logging.StreamHandler()
    ]
)
logger = logging.getLogger("app.streamlit.llm")

def initialize_llm() -> ChatOpenAI | None:
    """Initializes the ChatOpenAI client with strict type hinting and error handling."""
    # In production, prioritize Streamlit secrets or OS environment variables
    api_key: str | None = os.getenv("OPENAI_API_KEY")
    if not api_key:
        try:
            api_key = st.secrets.get("OPENAI_API_KEY")
        except FileNotFoundError:
            api_key = None

    if not api_key:
        logger.error("OPENAI_API_KEY is missing from environment and secrets.")
        return None
    
    try:
        llm = ChatOpenAI(
            model="gpt-3.5-turbo",
            temperature=0.7,
            api_key=api_key,
        )
        logger.info("ChatOpenAI client initialized successfully.")
        return llm
    except Exception as e:
        logger.error(f"Failed to initialize ChatOpenAI: {e}")
        return None

def generate_response(llm: ChatOpenAI, user_prompt: str) -> str:
    """Generates a response from the LLM based on user input, logging the interaction."""
    # Log the prompt securely (truncate if necessary to avoid log bloat/PII issues)
    logger.info(f"Received user prompt: {user_prompt[:50]}...") 
    
    messages = [
        SystemMessage(content="You are a helpful AI assistant."),
        HumanMessage(content=user_prompt)
    ]
    
    try:
        response = llm.invoke(messages)
        logger.info("Successfully generated response from LLM.")
        return str(response.content)
    except Exception as e:
        logger.error(f"Error during LLM invocation: {e}")
        return "An error occurred while generating the response. Please try again later."

def main() -> None:
    """Main Streamlit application entry point."""
    st.set_page_config(page_title="Day 7: LLM App", page_icon="🤖")
    st.title("LangChain & Streamlit Cloud App")
    
    llm = initialize_llm()
    if not llm:
        st.error("Application configuration error. Check logs for details.")
        st.stop()

    # Initialize chat history in session state to survive Streamlit reruns
    if "messages" not in st.session_state:
        st.session_state.messages = []

    # Display historical chat messages
    for message in st.session_state.messages:
        with st.chat_message(message["role"]):
            st.markdown(message["content"])

    # Handle new user input
    if user_prompt := st.chat_input("What is on your mind?"):
        # Display user message instantly
        st.chat_message("user").markdown(user_prompt)
        st.session_state.messages.append({"role": "user", "content": user_prompt})

        # Generate and display assistant response with a loading spinner
        with st.chat_message("assistant"):
            with st.spinner("Thinking..."):
                response_content = generate_response(llm, user_prompt)
                st.markdown(response_content)
        
        st.session_state.messages.append({"role": "assistant", "content": response_content})

if __name__ == "__main__":
    # Note: When running in a Jupyter Notebook, Streamlit apps won't render inline easily.
    # This code is meant to be extracted to a separate file (e.g., app.py).
    pass


## 🛠️ Practical Lab / Homework

**Your Task:** Deploy the application above to Streamlit Cloud.

1. **Extract Code:** Create a new local directory. Inside it, create a file named `app.py` and paste the Python code from the cell above into it.
2. **Define Dependencies:** Create a `requirements.txt` file in the same directory. Add the following dependencies:
   ```text
   streamlit
   langchain-core
   langchain-openai
   ```
3. **Version Control:** Initialize a Git repository, commit `app.py` and `requirements.txt`, and push to a new public or private repository on GitHub.
4. **Deploy:** Go to [share.streamlit.io](https://share.streamlit.io), sign in with GitHub, and click "New app". Select your repository, branch, and `app.py` as the main file path.
5. **Configure Secrets:** Before the app finishes booting, go to the app's Settings -> Secrets on the Streamlit dashboard and add your API key:
   ```toml
   OPENAI_API_KEY="sk-..."
   ```
6. **Verify Logs:** Once deployed, interact with the app. Then, click "Manage app" in the bottom right corner of your Streamlit Cloud deployment to view the terminal logs. Verify that your `INFO` and `ERROR` logs are appearing correctly.

---

## ⚠️ Common Pitfalls

When moving from local scripts to hosted Streamlit environments, watch out for:

1. **Hardcoding Secrets:** Storing API keys directly in `app.py` is a massive security risk, especially if pushed to GitHub. Always use `os.getenv()` or `st.secrets` as demonstrated.
2. **Ignoring UI Feedback:** AI API calls take time (latency). Failing to use `st.spinner()` or `st.write_stream()` makes the app feel frozen, leading users to spam the submit button.
3. **Uncontrolled Reruns:** Streamlit executes the entire script top-to-bottom on *every* user interaction. Failing to store chat history in `st.session_state` means the app will "forget" the conversation every time the user sends a new message.
4. **Silent Failures:** Without proper `try/except` blocks and logging, API errors (like rate limits or invalid keys) will either fail silently (freezing the app) or dump raw stack traces into the UI, exposing backend details to the user.

